# LizyML Tutorial: Regression with LightGBM

This notebook demonstrates the full LizyML workflow on the **Diamonds** dataset:

1. Data preparation (download from URL, includes categorical columns)
2. Config setup
3. Model fit (5-fold CV with early stopping)
4. Evaluate — metrics table + learning curve
5. Residual distribution
6. Feature importance — Split (plot + table)
7. Feature importance — Gain (plot + table)
8. Feature importance — SHAP (plot + table)

**Dataset**: Diamonds (~54,000 rows)  
**Target**: `price` (USD)  
**Categorical features**: `cut`, `color`, `clarity`

## 1. Setup

In [ ]:
from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lizyml import Model

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 2. Data Preparation

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"

df = pd.read_csv(URL)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
# Categorical columns that LizyML will auto-detect and encode
cat_cols = ["cut", "color", "clarity"]
for col in cat_cols:
    print(f"{col}: {sorted(df[col].unique())}")

In [ ]:
df.describe()

## 3. Config

In [ ]:
config = {
    "config_version": 1,
    "task": "regression",
    "data": {
        "target": "price",
    },
    "split": {
        "method": "kfold",
        "n_splits": 5,
        "random_state": 42,
        "shuffle": True,
    },
    "model": {
        "name": "lgbm",
        "params": {
            "n_estimators": 1000,
            "learning_rate": 0.05,
            "num_leaves": 63,
            "min_child_samples": 20,
            "verbose": -1,
        },
    },
    "features": {
        "auto_categorical": True,  # auto-detects cut/color/clarity
    },
    "training": {
        "seed": 42,
        "early_stopping": {
            "enabled": True,
            "rounds": 50,
            "inner_valid": {"method": "holdout", "ratio": 0.1},
        },
    },
    "evaluation": {
        "metrics": ["mae", "rmse", "r2", "mape", "huber"],
    },
}

## 4. Model Fit

In [ ]:
model = Model(config)
fit_result = model.fit(data=df)
print("Fit complete.")

## 5. Evaluate

### 5.1 Metrics Table

In [ ]:
eval_result = model.evaluate()
raw = eval_result["raw"]

# Build comparison table: OOF vs In-Fold (mean)
oof_metrics = raw["oof"]
if_metrics = raw["if_mean"]

metrics_df = pd.DataFrame(
    {"OOF": oof_metrics, "In-Fold (mean)": if_metrics}
).rename_axis("metric")

metrics_df.round(4)

### 5.2 Learning Curve

In [ ]:
fig = model.plot_learning_curve()
plt.show()

## 6. Residual Distribution

In [ ]:
y_true = df["price"].to_numpy(dtype=float)
residuals = y_true - fit_result.oof_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of residuals
axes[0].hist(residuals, bins=100, edgecolor="none", alpha=0.7)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5, label="zero")
axes[0].set_xlabel("Residual (actual − predicted)")
axes[0].set_ylabel("Count")
axes[0].set_title("OOF Residual Distribution")
axes[0].legend()

# Actual vs Predicted
axes[1].scatter(fit_result.oof_pred, y_true, alpha=0.1, s=2)
lim = [min(y_true.min(), fit_result.oof_pred.min()),
       max(y_true.max(), fit_result.oof_pred.max())]
axes[1].plot(lim, lim, "r--", linewidth=1.5, label="perfect")
axes[1].set_xlabel("Predicted price")
axes[1].set_ylabel("Actual price")
axes[1].set_title("Actual vs Predicted (OOF)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance: Split

In [ ]:
fig = model.importance_plot(kind="split")
plt.show()

In [ ]:
imp_split = (
    pd.Series(model.importance(kind="split"), name="importance_split")
    .sort_values(ascending=False)
    .to_frame()
)
imp_split

## 8. Feature Importance: Gain

In [ ]:
fig = model.importance_plot(kind="gain")
plt.show()

In [ ]:
imp_gain = (
    pd.Series(model.importance(kind="gain"), name="importance_gain")
    .sort_values(ascending=False)
    .to_frame()
)
imp_gain

## 9. Feature Importance: SHAP

Uses LightGBM's TreeSHAP algorithm (via `lizyml`'s `predict(return_shap=True)`).
SHAP values have shape `(n_samples, n_features)`.  
Importance = mean of absolute SHAP values per feature.

In [ ]:
X = df.drop(columns=["price"])
pred_result = model.predict(X, return_shap=True)

shap_values = pred_result.shap_values       # (n_samples, n_features)
feature_names = pred_result.used_features

print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# Table: mean |SHAP| per feature
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
shap_df = (
    pd.DataFrame({"mean_abs_shap": mean_abs_shap}, index=feature_names)
    .sort_values("mean_abs_shap", ascending=False)
)
shap_df

In [ ]:
# Plot: horizontal bar chart (top 20)
top = shap_df.head(20)
fig, ax = plt.subplots(figsize=(8, max(4, len(top) * 0.4)))
ax.barh(top.index[::-1], top["mean_abs_shap"][::-1])
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Feature Importance (SHAP)")
plt.tight_layout()
plt.show()